# Transcribing audio and video with Whisper

A simple Python pipeline for downloading a media file, transcribing it with Whisper, and exporting the results.

This notebook is designed for Google Colab. The source file must be available at a public URL that points directly to an MP4, MP3, WAV, M4A, or similar media file.


## 1. Install Whisper and FFmpeg


In [1]:
!pip -q install -U openai-whisper

!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)


## 2. Check for a GPU

Whisper can run on a CPU, but a GPU can make transcription much faster, especially with larger models.

In Colab, select Runtime > Change runtime type and choose a GPU if one is available.


In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cpu


## 3. Download a media file

Set MEDIA_URL to a public URL that points directly to a media file. The example below uses a video hosted by the Internet Archive.


In [4]:
from pathlib import Path
from urllib.request import urlretrieve

MEDIA_URL = "https://archive.org/download/tobacco_wbr62a00/tobacco_wbr62a00.mp4"
SOURCE_FILE = Path("tobacco_wbr62a00.mp4")

urlretrieve(MEDIA_URL, SOURCE_FILE)
print(SOURCE_FILE, SOURCE_FILE.stat().st_size, "bytes")


tobacco_wbr62a00.mp4 26535165 bytes


## 4. Choose a Whisper model

A model is the trained neural network used to recognize speech. Whisper provides models of different sizes.

Smaller models such as tiny and base are faster and use less memory. "small", "medium", and "large" generally require more computation but can improve accuracy. turbo is a faster version of the large model intended for transcription.

Models ending in .en, such as "tiny.en", are English-only. Other models are multilingual.

If your video/audio file is short and you don't want to (or are unable to) use GPUS, "tiny.en" or "base.en" is usually convenient. With a GPU, you might want to initially try "small" or "turbo".


In [5]:
import whisper

print(whisper.available_models())


['tiny.en', 'tiny', 'base.en', 'base', 'small.en', 'small', 'medium.en', 'medium', 'large-v1', 'large-v2', 'large-v3', 'large', 'large-v3-turbo', 'turbo']


In [6]:
MODEL_NAME = "tiny.en"

model = whisper.load_model(MODEL_NAME, device=device)
print("Loaded:", MODEL_NAME)


Loaded: tiny.en


## 5. Transcribe the file

Whisper uses FFmpeg to read the audio track from the media file. The result includes the transcript and timestamped segments.


In [7]:
import time

start = time.time()

result = model.transcribe(
    str(SOURCE_FILE),
    fp16=(device == "cuda")
)

elapsed = time.time() - start
print(f"Transcription time: {elapsed:.1f} seconds")


Transcription time: 43.5 seconds


## 6. View the transcript


In [10]:
import textwrap

print(textwrap.fill(result["text"], width=100))


 Think smoking makes you look cool. No way. What are you looking at? Hey, Christy. What? Think
smoking makes you look cool. No way. What? People don't want to think they're kids now and stop me
now. Always telling us what to do or not to do. But we know it's going on. We listen. Like smoking.
We don't know the reason it's not to. But out here comes up. That's when we meet the real decision.
And for us, it's not for a mom. And that's not even the hardest decision out there, but me. I mean,
you know? So how about a little cat and protein? My parents. They always think I'm not listening. I
hear them. Wait a minute. You know, every day you have to deal with stuff on your own. So you want
to know why I've decided not to smoke. Which reason would you like? I don't need to smoke to prove
myself. My coolness is not on trial here. I don't smoke because sometimes it's what you don't do.
That makes you who you are. These are the kids that I hang out with. They're all cool in different
ways. But w

## 7. View timestamped segments

The transcript is structured data, not just a block of text. Each segment includes start and end times.


In [11]:
for segment in result["segments"][:10]:
    print(
        f'{segment["start"]:7.1f} - {segment["end"]:7.1f}  '
        f'{segment["text"].strip()}'
    )


   30.0 -    32.0  Think smoking makes you look cool.
   35.1 -    36.1  No way.
   36.7 -    37.9  What are you looking at?
   44.8 -    46.0  Hey, Christy.
   61.0 -    62.0  What?
   63.0 -    65.0  Think smoking makes you look cool.
   67.0 -    68.0  No way.
   70.0 -    71.0  What?
   79.0 -    82.0  People don't want to think they're kids now and stop me now.
   82.0 -    84.0  Always telling us what to do or not to do.


## 8. Put the segments in a table


In [12]:
import pandas as pd

segments = pd.DataFrame([
    {
        "start": segment["start"],
        "end": segment["end"],
        "text": segment["text"].strip()
    }
    for segment in result["segments"]
])

segments.head(10)


,start,end,text
0,30.00,32.00,Think smoking makes you look cool.
1,35.12,36.12,No way.
2,36.68,37.88,What are you looking at?
3,44.84,46.04,"Hey, Christy."
4,61.00,62.00,What?
5,63.00,65.00,Think smoking makes you look cool.
6,67.00,68.00,No way.
7,70.00,71.00,What?
8,79.00,82.00,People don't want to think they're kids now an...
9,82.00,84.00,Always telling us what to do or not to do.


## 9. Export the results

Save the plain transcript, timestamped segments, and full Whisper result.


## Next steps

## Large Scale Analysis

The same pipeline can be extended to process multiple files, create searchable transcripts, or build pipelines that classify documents and analyze sentiment at scale.

For example, machine learning models are more likely to classify CDC produced anti-smoking ads as health-related and expressing a negative sentiment. Industry produced ads are likely to be primarily classified under non-health related topics and express a more generally positive sentiment.

## Carefully Consider Bias

I always emphasize this when discussing AI or ML related preparation of data sets. Cleaning data often means transforming it. For example, we used an english language model here. Does your audio file contain multiple languages, speech patterns, or variants of a language? Does some of your media transcribe at a much higher level of accuracy than other records in your collection? Is there a risk of data loss unevenly distributed across all media types? You may elevate some perspectives and and diminish others, allowing technical limitations to introduce a notable bias into your dataset.  